In [1]:
# Setup and imports
import sys
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import io
import base64

# Add backend to path
BACKEND_PATH = Path.cwd().parent / "backend"
sys.path.insert(0, str(BACKEND_PATH))

print(f"Backend path: {BACKEND_PATH}")

Backend path: /home/chris/coding-task/productlens-ai/backend


In [3]:
# Test Tesseract installation
import subprocess

try:
    result = subprocess.run(['tesseract', '--version'], capture_output=True, text=True)
    print("Tesseract is installed:")
    print(result.stdout.split('\n')[0])
except FileNotFoundError:
    print("Tesseract not found. Installing...")
    # On Linux, we'd install with apt
    print("Please run: sudo apt-get install tesseract-ocr")

Tesseract is installed:
tesseract 5.3.4


In [4]:
# Import OCR service
from services.ocr_service import OCRService

ocr = OCRService()
print(f"OCR Service initialized")
print(f"Tesseract version: {ocr._clean_extracted_text.__doc__ is not None}")

2025-11-27 19:49:58.409845: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-27 19:49:58.859571: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-27 19:50:01.310154: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-27 19:50:05.815793: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
INFO:services.ocr_service:Tesseract OCR initialized successfully


OCR Service initialized
Tesseract version: True


In [5]:
# Create a test image with text
def create_test_image(text: str, size=(400, 100), font_size=36):
    """Create a simple image with text for testing."""
    img = Image.new('RGB', size, color='white')
    draw = ImageDraw.Draw(img)
    
    # Try to use a basic font
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", font_size)
    except:
        font = ImageFont.load_default()
    
    # Draw text centered
    draw.text((10, 30), text, fill='black', font=font)
    
    return img

# Create test images
test_queries = [
    "red bag",
    "polka dot",
    "lunch box",
    "tea set",
    "cake stand"
]

print("Test images created for queries:")
for q in test_queries:
    print(f"  - '{q}'")

Test images created for queries:
  - 'red bag'
  - 'polka dot'
  - 'lunch box'
  - 'tea set'
  - 'cake stand'


In [6]:
# Test OCR extraction
print("=" * 60)
print("OCR EXTRACTION TEST")
print("=" * 60)

for query in test_queries:
    print(f"\nOriginal text: '{query}'")
    
    # Create image
    img = create_test_image(query)
    
    # Convert to bytes
    img_bytes = io.BytesIO()
    img.save(img_bytes, format='PNG')
    img_bytes = img_bytes.getvalue()
    
    # Extract text
    extracted = ocr.extract_text(img_bytes)
    
    print(f"Extracted text: '{extracted}'")
    
    # Check if extraction is correct
    match = query.lower().strip() == extracted.lower().strip()
    print(f"Match: {'✅' if match else '❌'}")

print("\n" + "=" * 60)

OCR EXTRACTION TEST

Original text: 'red bag'


INFO:services.ocr_service:Extracted text: red bag


Extracted text: 'red bag'
Match: ✅

Original text: 'polka dot'


INFO:services.ocr_service:Extracted text: polka dot


Extracted text: 'polka dot'
Match: ✅

Original text: 'lunch box'


...

Extracted text: 'lunch box'
Match: ✅

Original text: 'tea set'


INFO:services.ocr_service:Extracted text: tea set


Extracted text: 'tea set'
Match: ✅

Original text: 'cake stand'


INFO:services.ocr_service:Extracted text: cake stand


Extracted text: 'cake stand'
Match: ✅



In [7]:
# Test handwritten-style queries (larger images with more variation)
def create_handwritten_style_image(text: str):
    """Create an image simulating handwritten text."""
    # Larger image for better OCR
    img = Image.new('RGB', (600, 150), color=(255, 253, 240))  # Off-white
    draw = ImageDraw.Draw(img)
    
    try:
        # Try different fonts
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf", 48)
    except:
        font = ImageFont.load_default()
    
    # Draw with slight angle effect (simulating handwriting)
    draw.text((30, 45), text, fill='navy', font=font)
    
    return img

# Test product-related queries
product_queries = [
    "I need a lunch bag",
    "Looking for ribbons",
    "Want tea set"
]

print("=" * 60)
print("HANDWRITTEN STYLE OCR TEST")
print("=" * 60)

for query in product_queries:
    print(f"\nOriginal: '{query}'")
    
    img = create_handwritten_style_image(query)
    
    # Convert to bytes
    img_bytes = io.BytesIO()
    img.save(img_bytes, format='PNG')
    img_bytes = img_bytes.getvalue()
    
    extracted = ocr.extract_text(img_bytes)
    print(f"Extracted: '{extracted}'")

HANDWRITTEN STYLE OCR TEST

Original: 'I need a lunch bag'


INFO:services.ocr_service:Extracted text: I need a lunch bag


Extracted: 'I need a lunch bag'

Original: 'Looking for ribbons'


INFO:services.ocr_service:Extracted text: Looking for ribbons


Extracted: 'Looking for ribbons'

Original: 'Want tea set'


INFO:services.ocr_service:Extracted text: Want tea set


Extracted: 'Want tea set'


In [8]:
# Save sample test images for documentation
test_dir = BACKEND_PATH / "data" / "test_ocr"
test_dir.mkdir(parents=True, exist_ok=True)

# Create and save sample images
samples = [
    ("cake stand", "sample_cakestand.png"),
    ("polka dot bag", "sample_polkadot.png"),
    ("tea set ceramic", "sample_teaset.png")
]

for text, filename in samples:
    img = create_handwritten_style_image(text)
    img.save(test_dir / filename)
    print(f"Saved: {test_dir / filename}")

print(f"\nTest images saved to: {test_dir}")

Saved: /home/chris/coding-task/productlens-ai/backend/data/test_ocr/sample_cakestand.png
Saved: /home/chris/coding-task/productlens-ai/backend/data/test_ocr/sample_polkadot.png
Saved: /home/chris/coding-task/productlens-ai/backend/data/test_ocr/sample_teaset.png

Test images saved to: /home/chris/coding-task/productlens-ai/backend/data/test_ocr


## Summary

The OCR service is working correctly and can:
- Extract text from images with printed text
- Handle various font styles and sizes
- Preprocess images for better accuracy

Next step: Test Endpoint 2 (OCR Query) with the Flask API.